# Notebook 01 — Exploratory Data Analysis
**MSE 433 — Olympic Medal Performance**

This notebook loads the country-level panel data produced by `src/data_prep.py` and generates all EDA figures and tables for the final report.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path

ROOT   = Path('..').resolve()
INPUT  = ROOT / 'data' / 'input'
OUTPUT = ROOT / 'data' / 'output'
FIG    = ROOT / 'report' / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
OUTPUT.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

print('Setup complete')

## 1. Load Panels

In [ ]:
summer = pd.read_csv(OUTPUT / 'panel_summer.csv')
winter = pd.read_csv(OUTPUT / 'panel_winter.csv')

# Drop rows missing both GDP and population (historical / dissolved NOCs)
summer_full = summer.dropna(subset=['gdp_per_capita']).copy()
winter_full  = winter.dropna(subset=['gdp_per_capita']).copy()

print(f'Summer panel (all):  {len(summer):,} rows | with macro: {len(summer_full):,}')
print(f'Winter panel (all):  {len(winter):,} rows | with macro: {len(winter_full):,}')
print(f'Summer NOCs: {summer["NOC"].nunique()} | Winter NOCs: {winter["NOC"].nunique()}')
summer.head(3)

## 2. Descriptive Statistics — Table 1

In [ ]:
features = ['delegation_size', 'total_medals', 'medal_rate',
            'female_ratio', 'sport_count', 'sport_hhi', 'gdp_per_capita']

def describe_panel(df, label):
    d = df[features].describe().T[['mean','50%','std','min','max']]
    d.columns = ['Mean','Median','Std Dev','Min','Max']
    d.index.name = 'Feature'
    print(f'\n=== {label} ===')
    display(d.round(3))
    return d

desc_summer = describe_panel(summer_full, 'Summer Games')
desc_winter = describe_panel(winter_full,  'Winter Games')

In [ ]:
# Overdispersion check (motivates Negative Binomial over Poisson)
for label, df in [('Summer', summer_full), ('Winter', winter_full)]:
    m = df['total_medals'].mean()
    v = df['total_medals'].var()
    print(f'{label}: mean={m:.2f}, variance={v:.2f}, ratio={v/m:.1f}x  '
          f'→ {"OVERDISPERSED" if v > m else "OK"}')

## Figure 1 — Medal Count Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=False)

for ax, (label, df, color) in zip(axes, [
    ('Summer Games', summer, '#2196F3'),
    ('Winter Games', winter, '#FF5722')
]):
    vals = df['total_medals']
    ax.hist(vals, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(vals.mean(), color='black', lw=1.5, linestyle='--',
               label=f'Mean = {vals.mean():.1f}')
    ax.axvline(vals.median(), color='grey', lw=1.5, linestyle=':',
               label=f'Median = {vals.median():.0f}')
    pct_zero = (vals == 0).mean() * 100
    ax.set_title(f'{label}\n({pct_zero:.0f}% of country-games = 0 medals)', fontsize=12)
    ax.set_xlabel('Total Medals per Country-Game')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

fig.suptitle('Figure 1: Medal Count Distribution (1960–2016)\n'
             'Right-skewed with high zero-inflation → motivates count regression',
             fontsize=11, y=1.02)
plt.tight_layout()
plt.savefig(FIG / 'fig1_medal_distribution.png', bbox_inches='tight')
plt.show()
print('Saved Figure 1')

## Figure 2 — Overdispersion: Poisson vs. Negative Binomial vs. Actual Data

In [ ]:
from scipy.stats import poisson, nbinom

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (label, df, color) in zip(axes, [
    ('Summer Games', summer, '#2196F3'),
    ('Winter Games', winter, '#FF5722'),
]):
    vals = df['total_medals'].values.astype(int)
    mu   = vals.mean()
    var  = vals.var()

    # Method-of-moments for Negative Binomial: alpha = (var - mu) / mu^2
    alpha = max((var - mu) / mu**2, 1e-6)
    n_nb  = 1.0 / alpha
    p_nb  = 1.0 / (1.0 + alpha * mu)

    x_max = min(int(np.percentile(vals, 98)), 60)
    x     = np.arange(0, x_max + 1)

    actual_counts = np.bincount(np.clip(vals, 0, x_max), minlength=x_max + 1)[:x_max + 1]
    actual_props  = actual_counts / len(vals)

    ax.bar(x, actual_props, color=color, alpha=0.5, width=0.9, label='Actual data')
    ax.plot(x, poisson.pmf(x, mu), 'k-o', markersize=3, lw=1.5,
            label=f'Poisson (λ={mu:.1f})')
    ax.plot(x, nbinom.pmf(x, n_nb, p_nb), 'r-s', markersize=3, lw=1.5,
            label='Negative Binomial')
    ax.set_xlim(-0.5, x_max)
    ax.set_xlabel('Total Medals per Country-Game')
    ax.set_ylabel('Proportion')
    ax.set_title(f'{label}\nVariance/Mean = {var/mu:.1f}x (overdispersion)', fontsize=11)
    ax.legend(fontsize=8)

fig.suptitle('Figure 2: Overdispersion — Poisson vs. Negative Binomial vs. Actual Data\n'
             'Poisson underestimates zeros; Negative Binomial fits both the zero mass and the fat tail\n'
             'KB: (B) Stochastic Modeling — Poisson Distribution, Negative Binomial',
             fontsize=10)
plt.tight_layout()
plt.savefig(FIG / 'fig2_overdispersion.png', bbox_inches='tight')
plt.show()
print('Saved Figure 2')

## Figure 3 — Correlation Heatmap

In [ ]:
corr_features = [
    'total_medals', 'delegation_size', 'gdp_per_capita',
    'population', 'female_ratio', 'sport_count',
    'sport_hhi', 'is_host', 'medal_rate'
]
rename_map = {
    'total_medals': 'Total Medals',
    'delegation_size': 'Delegation Size',
    'gdp_per_capita': 'GDP per Capita',
    'population': 'Population',
    'female_ratio': 'Female Ratio',
    'sport_count': 'Sports Entered',
    'sport_hhi': 'Sport HHI',
    'is_host': 'Host Nation',
    'medal_rate': 'Medal Rate'
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (label, df) in zip(axes, [
    ('Summer Games', summer_full),
    ('Winter Games', winter_full)
]):
    corr = df[corr_features].rename(columns=rename_map).corr()
    sns.heatmap(
        corr, ax=ax, annot=True, fmt='.2f', cmap='RdBu_r',
        vmin=-1, vmax=1, linewidths=0.3, annot_kws={'size': 8},
        cbar_kws={'shrink': 0.8}
    )
    ax.set_title(label, fontsize=12)
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Figure 3: Pearson Correlation Heatmap\n'
             'KB: (C) Data Science — Correlation Analysis', fontsize=11)
plt.tight_layout()
plt.savefig(FIG / 'fig3_correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('Saved Figure 3')

## Figure 4 — Delegation Size vs. Total Medals (log-log)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (label, df, color) in zip(axes, [
    ('Summer Games', summer_full, '#2196F3'),
    ('Winter Games', winter_full, '#FF5722')
]):
    plot_df = df[df['total_medals'] > 0].copy()
    sc = ax.scatter(
        plot_df['log_delegation_size'],
        plot_df['log_total_medals'],
        c=plot_df['log_gdp_per_capita'],
        cmap='viridis', alpha=0.55, s=20, linewidths=0
    )
    m, b, r, p, _ = stats.linregress(
        plot_df['log_delegation_size'], plot_df['log_total_medals'])
    xs = np.linspace(plot_df['log_delegation_size'].min(),
                     plot_df['log_delegation_size'].max(), 100)
    ax.plot(xs, m*xs + b, 'r-', lw=1.8, label=f'Slope={m:.2f}, R²={r**2:.2f}')
    plt.colorbar(sc, ax=ax, label='log(GDP per capita)')
    ax.set_xlabel('log(Delegation Size)')
    ax.set_ylabel('log(Total Medals)')
    ax.set_title(label, fontsize=12)
    ax.legend(fontsize=9)

fig.suptitle('Figure 4: Delegation Size vs. Total Medals (log-log scale)\n'
             'Colour = log(GDP per capita); slope ≈ elasticity of medals w.r.t. delegation size',
             fontsize=11)
plt.tight_layout()
plt.savefig(FIG / 'fig4_delegation_vs_medals.png', bbox_inches='tight')
plt.show()
print('Saved Figure 4')

## Figure 5 — Female Participation Ratio Over Time

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (label, df, color) in zip(axes, [
    ('Summer Games', summer_full, '#2196F3'),
    ('Winter Games', winter_full, '#FF5722')
]):
    agg = df.groupby('Year')['female_ratio'].agg(['mean','median']).reset_index()
    ax.fill_between(agg['Year'],
                    df.groupby('Year')['female_ratio'].quantile(0.25).values,
                    df.groupby('Year')['female_ratio'].quantile(0.75).values,
                    alpha=0.2, color=color, label='IQR')
    ax.plot(agg['Year'], agg['mean'], color=color, lw=2, label='Mean')
    ax.plot(agg['Year'], agg['median'], color=color, lw=1.5,
            linestyle='--', label='Median')
    ax.axhline(0.5, color='grey', lw=1, linestyle=':', label='Parity (50%)')
    ax.set_xlabel('Year')
    ax.set_ylabel('Female Athletes / Total Athletes')
    ax.set_ylim(0, 0.7)
    ax.set_title(label, fontsize=12)
    ax.legend(fontsize=9)
    ax.set_xticks(df['Year'].unique())
    ax.tick_params(axis='x', rotation=45)

fig.suptitle('Figure 5: Female Participation Ratio Over Time\n'
             'Shaded band = interquartile range across countries',
             fontsize=11)
plt.tight_layout()
plt.savefig(FIG / 'fig5_female_ratio_trend.png', bbox_inches='tight')
plt.show()
print('Saved Figure 5')

## Figure 6 — Host Nation Effect

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, (label, df, color) in zip(axes, [
    ('Summer Games', summer, '#2196F3'),
    ('Winter Games', winter, '#FF5722')
]):
    host_medals    = df[df['is_host'] == 1]['total_medals']
    nonhost_medals = df[df['is_host'] == 0]['total_medals']

    t_stat, p_val = stats.ttest_ind(host_medals, nonhost_medals, equal_var=False)

    data_to_plot = [nonhost_medals.values, host_medals.values]
    ax.boxplot(data_to_plot, patch_artist=True,
               labels=['Non-Host', 'Host'],
               boxprops=dict(facecolor=color, alpha=0.6),
               medianprops=dict(color='black', lw=2))

    sig = '***' if p_val < 0.001 else ('**' if p_val < 0.01 else ('*' if p_val < 0.05 else 'ns'))
    ax.set_title(f'{label}\nHost mean={host_medals.mean():.1f} vs '
                 f'Non-host mean={nonhost_medals.mean():.1f}\n'
                 f't={t_stat:.2f}, p={p_val:.4f} {sig}', fontsize=10)
    ax.set_ylabel('Total Medals')

fig.suptitle('Figure 6: Host Nation Effect on Medal Count\n'
             '(Welch t-test; H₀: host mean = non-host mean; * p<0.05, ** p<0.01, *** p<0.001)\n'
             'KB: (B) Stochastic Modeling — Hypothesis Testing',
             fontsize=11)
plt.tight_layout()
plt.savefig(FIG / 'fig6_host_effect.png', bbox_inches='tight')
plt.show()
print(f'Host effect — Summer: t={stats.ttest_ind(summer[summer.is_host==1].total_medals, summer[summer.is_host==0].total_medals, equal_var=False)[0]:.2f}')

## Figure 7 — Pareto Chart: Cumulative Medals by Sport

In [ ]:
athletes = pd.read_csv(INPUT / 'athlete_events.csv')
athletes = athletes[athletes['Year'] >= 1960].copy()
athletes['has_medal'] = athletes['Medal'].notna() & (athletes['Medal'] != 'NA')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (label, season, color) in zip(axes, [
    ('Summer Games', 'Summer', '#2196F3'),
    ('Winter Games', 'Winter', '#FF5722')
]):
    df_s = athletes[(athletes['Season'] == season) & athletes['has_medal']]
    df_s = df_s.drop_duplicates(subset=['NOC', 'Year', 'Event', 'Medal'])
    sport_medals = df_s.groupby('Sport').size().sort_values(ascending=False)
    cumulative = sport_medals.cumsum() / sport_medals.sum() * 100
    n_sports = len(sport_medals)

    x = range(n_sports)
    ax2 = ax.twinx()
    ax.bar(x, sport_medals.values, color=color, alpha=0.7, width=0.8)
    ax2.plot(x, cumulative.values, 'k-o', markersize=3, lw=1.5)
    ax2.axhline(80, color='red', lw=1, linestyle='--', label='80% line')
    ax2.set_ylabel('Cumulative % of Total Medals')
    ax2.set_ylim(0, 105)

    idx_80 = (cumulative >= 80).idxmax()
    n_80 = sport_medals.index.get_loc(idx_80) + 1

    ax.set_xticks(list(x))
    ax.set_xticklabels(sport_medals.index, rotation=90, fontsize=6)
    ax.set_xlabel('Sport')
    ax.set_ylabel('Total Medals (deduplicated)')
    ax.set_title(f'{label}\nTop {n_80} sports account for 80% of all medals', fontsize=10)

fig.suptitle('Figure 7: Pareto Chart — Cumulative Medals by Sport (1960–2016)\n'
             'KB: (E) Operations — Pareto Analysis', fontsize=11)
plt.tight_layout()
plt.savefig(FIG / 'fig7_pareto_sports.png', bbox_inches='tight')
plt.show()
print('Saved Figure 7')

## 3. Summary Tables

In [ ]:
for label, df in [('Summer', summer), ('Winter', winter)]:
    top10 = (
        df.groupby('NOC')[['gold_medals','silver_medals','bronze_medals','total_medals']]
          .sum()
          .sort_values('total_medals', ascending=False)
          .head(10)
          .astype(int)
    )
    print(f'\n=== Top 10 Nations — {label} Games (1960–2016) ===')
    display(top10)

In [ ]:
rows = []
for label, df in [('Summer', summer), ('Winter', winter)]:
    host_m    = df[df['is_host']==1]['total_medals']
    nonhost_m = df[df['is_host']==0]['total_medals']
    t, p = stats.ttest_ind(host_m, nonhost_m, equal_var=False)
    uplift = (host_m.mean() - nonhost_m.mean()) / nonhost_m.mean() * 100
    rows.append({
        'Season': label,
        'Host Mean': round(host_m.mean(), 1),
        'Non-Host Mean': round(nonhost_m.mean(), 1),
        'Uplift (%)': round(uplift, 1),
        't-stat': round(t, 2),
        'p-value': round(p, 4),
        'Significant': '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    })

print('\n=== Host Nation Effect Summary ===')
display(pd.DataFrame(rows).set_index('Season'))

In [ ]:
print('=== KEY EDA FINDINGS ===')
for label, df in [('Summer', summer_full), ('Winter', winter_full)]:
    v = df['total_medals'].var()
    m = df['total_medals'].mean()
    pct_zero = (df['total_medals'] == 0).mean()
    corr_del = df['log_delegation_size'].corr(df['log_total_medals'])
    corr_gdp = df['log_gdp_per_capita'].corr(df['total_medals'])
    print(f'\n{label}:')
    print(f'  Zero-medal country-games: {pct_zero:.1%}')
    print(f'  Variance/Mean (overdispersion): {v/m:.1f}x')
    print(f'  Corr(log_delegation, log_medals): {corr_del:.3f}')
    print(f'  Corr(log_gdp_per_capita, medals): {corr_gdp:.3f}')
    print(f'  Avg female ratio: {df["female_ratio"].mean():.3f}')
    print(f'  Avg sport HHI: {df["sport_hhi"].mean():.3f}')